# The targeted latent hijack and its detectability: CODI 1B (ledger §136, Build 7)

**No training.** Victims are correctly answered multi-step GSM8K test questions. For each, an
attacker value near the gold penultimate intermediate, and the answer it implies, are fixed before
any write. A natural bank state whose carrier (thought 3) reads that value is written in and
the answer is decoded again; other arms write the store's K/V (position 4), the learned writer's
vector, or the decoy slot (5). Then the defender's side: per-slot
readouts after the write, the answer's confidence, linear detectors on the states, and verification
through the model's own explicit chain. Predictions H1–H7 are in the ledger. No dataset to attach.
Publish the output when finished.

## Setup

In [ ]:
REPO_URL = "https://github.com/0x0shephard/latent-reasoning.git"
RUN_COMMIT = "0000000000000000000000000000000000000000"
REPO_DIR = "/kaggle/working/latent-reasoning"
CONFIG = "configs/official_codi_llama1b.yaml"
MODEL = "codi1b"
PRECISION = "float16"
BATCH_SIZE = "32"
OUTPUT_ROOT = "/kaggle/working/hijack_codi1b"
CARRIER, STORE, DECOY, EXTRA = "3", "4", "5", []
RUN_SMOKE = True

import glob, json, os, pathlib, subprocess, sys, time
os.environ.setdefault("HF_HUB_DISABLE_XET", "1")
os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "300")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
assert len(RUN_COMMIT) == 40 and RUN_COMMIT != "0" * 40
if not pathlib.Path(REPO_DIR).exists():
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
subprocess.run(["git", "-C", REPO_DIR, "fetch", "--all", "--tags"], check=True)
subprocess.run(["git", "-C", REPO_DIR, "checkout", RUN_COMMIT], check=True)
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print("code commit", subprocess.run(["git", "rev-parse", "HEAD"], check=True, capture_output=True, text=True).stdout.strip())
print("gpu", subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip())

### Environment and focused tests

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "tpot"], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "transformers==4.52.4", "datasets==3.6.0", "peft==0.15.2", "accelerate==1.7.0",
                "huggingface-hub>=0.34,<1.0", "safetensors==0.5.3", "pytest>=8,<10"], check=True)
probe = subprocess.run([sys.executable, "-c",
    "from peft.import_utils import is_torchao_available\ntry:\n print(is_torchao_available())\nexcept ImportError:\n print('incompatible')"],
    capture_output=True, text=True)
if "incompatible" in probe.stdout:
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], check=True)
subprocess.run([sys.executable, "-m", "pytest", "-q", *["tests/test_official_codi_family.py", "tests/test_cache_carrier.py", "tests/test_latent_repair_runner.py", "tests/test_latent_hijack.py"]], check=True)

def run_with_retry(command, attempts=3, wait=60):
    for attempt in range(1, attempts + 1):
        result = subprocess.run(command)
        if result.returncode == 0:
            return
        if attempt == attempts:
            raise subprocess.CalledProcessError(result.returncode, command)
        print(f"runner exited with {result.returncode}; retrying in {wait}s ({attempt + 1}/{attempts})")
        time.sleep(wait)

## The hijack run (smoke first, then full)

In [ ]:
def hijack(out, smoke):
    cmd = [sys.executable, "-u", "scripts/run_codi_latent_hijack.py", "--config", CONFIG, "--model-key", MODEL, "--output-dir", out,
           "--carrier", CARRIER, "--store", STORE, "--decoy", DECOY, "--batch-size", BATCH_SIZE, "--precision", PRECISION, "--device", "cuda"]
    if EXTRA:
        cmd += ["--extra-carriers", *EXTRA]
    if smoke:
        cmd.append("--smoke")
    run_with_retry(cmd)
    return json.loads(open(pathlib.Path(out) / "summary.json").read())

if RUN_SMOKE:
    hijack(OUTPUT_ROOT + "_smoke", True)
summary = hijack(OUTPUT_ROOT, False)
import pandas as pd
from IPython.display import display
rep = summary["report"]
print(json.dumps(rep["victims"], indent=2)); print(json.dumps(summary["checks"], indent=2))
print("--- outcomes over attacked victims")
display(pd.DataFrame({k: {o: v[o]["rate"] for o in ("hijacked", "broken", "intact", "no_answer")} | {"n": v["hijacked"]["n"]} for k, v in rep["outcomes"]["attacked"].items()}).T)
print("--- readouts after the nearest-donor write (over hijacked victims): share reading the attacker's value / the own value")
display(pd.DataFrame({s: {"attacker": v["attacker"], "own": v["own"], "n": v["n"]} for s, v in rep["readings_after"]["state_nearest"]["slots"].items()}).T)
print("--- confidence:", json.dumps(rep["confidence"], indent=2))
print("--- linear detectors (out-of-fold AUROC, hijacked vs clean):", json.dumps(rep["detectors"]["hijacked_vs_clean"], indent=2))
print("--- verification through the explicit chain:", json.dumps(rep["verification"], indent=2))
if rep.get("decoy"):
    print("--- decoy write:", json.dumps(rep["decoy"], indent=2))
print("Reminder: publish this notebook's output as a dataset (Output tab -> New Dataset).")

## Keep the output under Kaggle's 500-file cap

In [ ]:
import shutil
shutil.rmtree(REPO_DIR, ignore_errors=True)   # the clone (with .git) is not an output; without this the run's own files can be dropped from the saved output
print("removed", REPO_DIR, "; remaining output files:", sum(len(f) for _, _, f in os.walk("/kaggle/working")))